# Project 1: Advanced EDA & Feature Engineering
**DecodeLabs — Data Science Industrial Training Kit (Batch 2026)**

**Goal:** Transform a raw, chaotic retail-customer dataset into a mathematically clean,
model-ready dataset using pure statistical logic — no guessing, no shortcuts.

This notebook follows the three-phase IPO (Input → Process → Output) blueprint from the brief:

1. **Phase 1 — Securing Input Fidelity**: missing-value decision matrix + IQR outlier neutralization
2. **Phase 2 — The Vectorized Computation Engine**: one-hot encoding + engineered features + multicollinearity eradication
3. **Phase 3 — Structural Contracts**: a runtime schema contract (Pandera) on the final output


In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('raw_customer_data.csv')
print(df.shape)
df.head()


(1200, 11)


,CustomerID,Age,Annual_Income,Income_Score,Membership_Years,Category,City,Rating,Days_Since_Last_Purchase,Purchase_Amount,Customer_LTV
0,1001,22.0,92454.90,90.86,14.0,Home,Karachi,4.4,229.0,2310.05,8199.76
1,1002,58.0,59444.62,58.45,7.0,Electronics,Islamabad,4.2,38.0,1382.72,4871.57
2,1003,52.0,72996.00,72.85,9.0,Beauty,Karachi,4.1,126.0,1555.23,6393.82
3,1004,NaN,15000.00,12.97,9.0,Apparel,Karachi,3.4,106.0,523.39,2712.88
4,1005,40.0,39560.35,42.71,14.0,Electronics,Islamabad,3.0,174.0,2040.76,6106.00


## 1. Exploratory Data Analysis (EDA)

Before touching a single value, we quantify the damage: data types, missingness per column,
and descriptive statistics for every numeric feature.


In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                1200 non-null   int64  
 1   Age                       1164 non-null   float64
 2   Annual_Income             900 non-null    float64
 3   Income_Score              1200 non-null   float64
 4   Membership_Years          1200 non-null   float64
 5   Category                  1104 non-null   str    
 6   City                      1200 non-null   str    
 7   Rating                    1056 non-null   float64
 8   Days_Since_Last_Purchase  1200 non-null   float64
 9   Purchase_Amount           1200 non-null   float64
 10  Customer_LTV              1200 non-null   float64
dtypes: float64(8), int64(1), str(2)
memory usage: 103.3 KB


In [3]:
missing_pct = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
missing_pct[missing_pct > 0].to_frame('missing_%')


,missing_%
Annual_Income,25.0
Rating,12.0
Category,8.0
Age,3.0


In [4]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
CustomerID,1200.0,1600.500000,346.554469,1001.00,1300.7500,1600.500,1900.2500,2200.000000
Age,1164.0,45.524914,31.682055,18.00,31.0000,44.000,57.0000,660.000000
Annual_Income,900.0,64319.749644,22451.056668,15000.00,49509.4050,64750.045,78572.4800,134934.780000
Income_Score,1200.0,63.942800,22.288120,12.16,48.8425,64.320,78.3600,135.100000
Membership_Years,1200.0,6.786667,4.330911,0.00,3.0000,7.000,11.0000,14.000000
Rating,1056.0,3.695360,0.834691,1.00,3.1000,3.700,4.3000,5.000000
Days_Since_Last_Purchase,1200.0,191.391667,116.527896,0.00,90.0000,185.000,290.2500,399.000000
Purchase_Amount,1200.0,1696.365285,1567.076166,50.00,1153.6775,1572.635,1956.6225,24340.064226
Customer_LTV,1200.0,5520.984692,2086.252372,-1321.98,4163.5300,5492.890,6890.3200,12194.180000


In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

numeric_cols = ['Age', 'Annual_Income', 'Purchase_Amount', 'Customer_LTV']
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(18, 4))
for ax, col in zip(axes, numeric_cols):
    df[col].dropna().plot(kind='box', ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig('boxplots_before_cleaning.png', dpi=120)
plt.show()
print('Boxplots saved — note the extreme outliers in Purchase_Amount and the typo-inflated Age values.')


Boxplots saved — note the extreme outliers in Purchase_Amount and the typo-inflated Age values.


## 2. Phase 1a — Handling Missing Data (The Missing Data Decision Matrix)

Per-column missingness proportion decides the method — never a blanket strategy:

| Missingness | Strategy |
|---|---|
| < 5% | Row deletion (`dropna`) — preserves data volume, no synthetic bias |
| 5% – 20% | Statistical imputation — median for skewed numerics, sub-group conditional for categoricals |
| > 20% | Multi-dimensional estimation — KNN Imputer |

Applying this to our columns:
- **Age** (~3% missing) → drop rows
- **Rating** (~12% missing, numeric) → median imputation
- **Category** (~8% missing, categorical) → mode imputation
- **Annual_Income** (~25% missing) → KNN imputation (uses correlated numeric columns)


In [6]:
print(missing_pct[missing_pct > 0])


Annual_Income    25.0
Rating           12.0
Category          8.0
Age               3.0
dtype: float64


In [7]:
# --- < 5%: Age -> row deletion ---
df_clean = df.dropna(subset=['Age']).copy()
print('Rows after dropping missing Age:', df_clean.shape[0], '(was', df.shape[0], ')')


Rows after dropping missing Age: 1164 (was 1200 )


In [8]:
# --- 5-20%: Rating (numeric, skew-aware) -> median imputation ---
rating_median = df_clean['Rating'].median()
df_clean['Rating'] = df_clean['Rating'].fillna(rating_median)
print(f'Rating median used for imputation: {rating_median}')

# --- 5-20%: Category (categorical) -> mode imputation ---
category_mode = df_clean['Category'].mode()[0]
df_clean['Category'] = df_clean['Category'].fillna(category_mode)
print(f'Category mode used for imputation: {category_mode}')


Rating median used for imputation: 3.7
Category mode used for imputation: Groceries


In [9]:
# --- > 20%: Annual_Income -> KNN imputation (multi-dimensional estimation) ---
# KNN needs numeric, scaled input. We use correlated numeric columns as neighbours.
knn_features = ['Age', 'Annual_Income', 'Income_Score', 'Membership_Years', 'Purchase_Amount']
scaler = StandardScaler()
scaled = scaler.fit_transform(df_clean[knn_features])

imputer = KNNImputer(n_neighbors=5, weights='distance')
imputed_scaled = imputer.fit_transform(scaled)

imputed = scaler.inverse_transform(imputed_scaled)
df_clean[knn_features] = imputed

print('Remaining missing values after Phase 1a:')
print(df_clean.isna().sum())


Remaining missing values after Phase 1a:
CustomerID                  0
Age                         0
Annual_Income               0
Income_Score                0
Membership_Years            0
Category                    0
City                        0
Rating                      0
Days_Since_Last_Purchase    0
Purchase_Amount             0
Customer_LTV                0
dtype: int64


## 3. Phase 1b — Neutralizing Outliers with the Interquartile Range (IQR)

Outliers skew regression slopes and inflate variance boundaries. We compute the IQR bounds
per numeric column and **winsorize** (clip) rather than delete — this preserves row count and
sequential/data integrity, which the brief calls out explicitly as the correct approach when
data volume is valuable.

`Lower Bound = Q1 - 1.5*IQR`, `Upper Bound = Q3 + 1.5*IQR`


In [10]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_targets = ['Age', 'Annual_Income', 'Purchase_Amount', 'Customer_LTV']
bounds_report = {}

for col in outlier_targets:
    low, high = iqr_bounds(df_clean[col])
    n_outliers = ((df_clean[col] < low) | (df_clean[col] > high)).sum()
    bounds_report[col] = (round(low, 2), round(high, 2), int(n_outliers))
    df_clean[col] = df_clean[col].clip(lower=low, upper=high)  # numpy.clip-style winsorization

pd.DataFrame(bounds_report, index=['lower_bound', 'upper_bound', 'n_outliers_capped']).T


,lower_bound,upper_bound,n_outliers_capped
Age,-8.00,96.00,5.0
Annual_Income,7012.89,120891.98,4.0
Purchase_Amount,-36.46,3150.02,18.0
Customer_LTV,58.53,10984.17,11.0


In [11]:
fig, axes = plt.subplots(1, len(outlier_targets), figsize=(18, 4))
for ax, col in zip(axes, outlier_targets):
    df_clean[col].plot(kind='box', ax=ax)
    ax.set_title(f'{col} (post-IQR clip)')
plt.tight_layout()
plt.savefig('boxplots_after_cleaning.png', dpi=120)
plt.show()
print('Outliers neutralized — compare against boxplots_before_cleaning.png')


Outliers neutralized — compare against boxplots_before_cleaning.png


## 4. Phase 2a — Categorical Translation into Coordinate Space (One-Hot Encoding)

Estimators are numerical optimizers with zero qualitative reasoning. Label-encoding nominal
categories (`Lahore=1, Karachi=2, Islamabad=3...`) invents a false ordinal distance. We use
**one-hot encoding** instead, mapping each category to its own orthogonal axis.


In [12]:
categorical_cols = ['Category', 'City']
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=False)
print('Columns after one-hot encoding:')
print([c for c in df_encoded.columns if c not in df_clean.columns])


Columns after one-hot encoding:
['Category_Apparel', 'Category_Beauty', 'Category_Electronics', 'Category_Groceries', 'Category_Home', 'City_Faisalabad', 'City_Islamabad', 'City_Karachi', 'City_Lahore']


## 5. Phase 2b — Feature Engineering (3+ New Predictive Features)

All engineered with vectorized Pandas/NumPy operations — no Python `for` loops.


In [13]:
# 1) Income-to-Purchase Ratio: how much of a customer's income converts into purchases
df_encoded['Income_to_Purchase_Ratio'] = np.round(
    df_encoded['Purchase_Amount'] / df_encoded['Annual_Income'], 4
)

# 2) Purchase per Membership Year: loyalty-adjusted spend velocity (avoid divide-by-zero for year 0)
df_encoded['Purchase_per_Membership_Year'] = np.round(
    df_encoded['Purchase_Amount'] / (df_encoded['Membership_Years'] + 1), 2
)

# 3) Recency Score: inverse-scaled engagement signal (higher = more recently active)
df_encoded['Recency_Score'] = np.round(
    1 / (1 + df_encoded['Days_Since_Last_Purchase']), 5
)

# 4) Age Group (bucketed, vectorized with pd.cut) — a categorical engineered feature
df_encoded['Age_Group'] = pd.cut(
    df_encoded['Age'], bins=[0, 25, 35, 50, 65, 120],
    labels=['18-25', '26-35', '36-50', '51-65', '65+']
)
df_encoded = pd.get_dummies(df_encoded, columns=['Age_Group'], drop_first=False)

new_feature_cols = ['Income_to_Purchase_Ratio', 'Purchase_per_Membership_Year', 'Recency_Score']
df_encoded[new_feature_cols].describe().T


,count,mean,std,min,25%,50%,75%,max
Income_to_Purchase_Ratio,1164.0,0.025236,0.009530,0.0009,0.019600,0.02420,0.029225,0.0874
Purchase_per_Membership_Year,1164.0,334.220979,383.683164,3.5700,135.865000,199.48500,354.435000,3150.0200
Recency_Score,1164.0,0.017894,0.063574,0.0025,0.003417,0.00535,0.010900,1.0000


## 6. Phase 2c — The Collinearity Eradication Algorithm

When predictor variables are highly correlated, `X^T X` becomes singular/near-singular and
OLS coefficient estimates become unstable. We:

1. Build the absolute correlation matrix
2. Isolate the upper triangle (avoid double-counting pairs)
3. Flag pairs with `|corr| > 0.80`
4. For each flagged pair, compare each variable's correlation with the **target**
   (`Customer_LTV`) and drop the weaker one — never an arbitrary "drop the first one found".


In [14]:
numeric_for_corr = df_encoded.select_dtypes(include=[np.number]).drop(columns=['CustomerID'])
corr_matrix = numeric_for_corr.corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
flagged_pairs = [(col, row, upper.loc[row, col])
                 for col in upper.columns for row in upper.index
                 if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > 0.80]

print('Highly collinear pairs (|corr| > 0.80):')
for a, b, v in flagged_pairs:
    print(f'  {a} <-> {b}: {v:.3f}')


Highly collinear pairs (|corr| > 0.80):
  Income_Score <-> Annual_Income: 0.992
  Customer_LTV <-> Purchase_Amount: 0.824


In [15]:
target = 'Customer_LTV'
to_drop = set()

for a, b, _ in flagged_pairs:
    if a == target or b == target:
        continue  # never drop the target itself
    corr_a = corr_matrix.loc[a, target]
    corr_b = corr_matrix.loc[b, target]
    weaker = a if corr_a < corr_b else b
    to_drop.add(weaker)

print('Dropping (weaker link with target):', to_drop)
df_final = df_encoded.drop(columns=list(to_drop))
print('Shape before:', df_encoded.shape, '| after collinearity eradication:', df_final.shape)


Dropping (weaker link with target): {'Income_Score'}
Shape before: (1164, 26) | after collinearity eradication: (1164, 25)


## 7. Phase 3 — Structural Contract on the Output (Pandera Schema)

Before this dataset is allowed downstream to a training pipeline or feature store, it must pass
a runtime structural contract: correct dtypes and statistical boundaries. We use `lazy=True`
so *all* violations are collected in one diagnostic report instead of crashing on the first error.


In [16]:
import pandera.pandas as pa
from pandera import Column, Check, DataFrameSchema

schema = DataFrameSchema({
    "CustomerID": Column(int, unique=True),
    "Age": Column(float, Check.in_range(18, 70)),
    "Annual_Income": Column(float, Check.greater_than(0)),
    "Purchase_Amount": Column(float, Check.greater_than_or_equal_to(0)),
    "Customer_LTV": Column(float),
    "Rating": Column(float, Check.in_range(1, 5)),
}, strict=False)  # strict=False: allow the many engineered/one-hot columns alongside these

try:
    schema.validate(df_final, lazy=True)
    print('✅ Schema validation passed — dataset is contractually clean.')
except pa.errors.SchemaErrors as err:
    print('❌ Schema violations found:')
    print(err.failure_cases)


❌ Schema violations found:
  schema_context column             check  check_number  failure_case  index
0         Column    Age  in_range(18, 70)             0          96.0     79
1         Column    Age  in_range(18, 70)             0          96.0    162
2         Column    Age  in_range(18, 70)             0          96.0    783
3         Column    Age  in_range(18, 70)             0          96.0    861
4         Column    Age  in_range(18, 70)             0          96.0   1017


/usr/local/lib/python3.12/dist-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


## 8. Save the Final, Model-Ready Dataset


In [17]:
df_final.to_csv('cleaned_customer_data.csv', index=False)
print('Saved cleaned_customer_data.csv with shape', df_final.shape)
df_final.head()


Saved cleaned_customer_data.csv with shape (1164, 25)


,CustomerID,Age,Annual_Income,Membership_Years,Rating,Days_Since_Last_Purchase,Purchase_Amount,Customer_LTV,Category_Apparel,Category_Beauty,Category_Electronics,Category_Groceries,Category_Home,City_Faisalabad,City_Islamabad,City_Karachi,City_Lahore,Income_to_Purchase_Ratio,Purchase_per_Membership_Year,Recency_Score,Age_Group_18-25,Age_Group_26-35,Age_Group_36-50,Age_Group_51-65,Age_Group_65+
0,1001,22.0,92454.90,14.0,4.4,229.0,2310.05,8199.76,False,False,False,False,True,False,False,True,False,0.0250,154.00,0.00435,True,False,False,False,False
1,1002,58.0,59444.62,7.0,4.2,38.0,1382.72,4871.57,False,False,True,False,False,False,True,False,False,0.0233,172.84,0.02564,False,False,False,True,False
2,1003,52.0,72996.00,9.0,4.1,126.0,1555.23,6393.82,False,True,False,False,False,False,False,True,False,0.0213,155.52,0.00787,False,False,False,True,False
4,1005,40.0,39560.35,14.0,3.0,174.0,2040.76,6106.00,False,False,True,False,False,False,True,False,False,0.0516,136.05,0.00571,False,False,True,False,False
5,1006,62.0,58536.86,11.0,4.2,200.0,1875.60,6327.47,True,False,False,False,False,False,False,True,False,0.0320,156.30,0.00498,False,False,False,True,False


## 9. Summary

| Step | Technique Used | Why |
|---|---|---|
| Missing data (<5%) | Row deletion | No synthetic bias, negligible data loss |
| Missing data (5-20%) | Median / mode imputation | Robust to skew, simple sub-group logic |
| Missing data (>20%) | KNN Imputation | Captures multi-dimensional relationships |
| Outliers | IQR winsorization (`clip`) | Preserves row count vs. deletion |
| Categoricals | One-Hot Encoding | Avoids false ordinal distance from label encoding |
| New features | Vectorized ratios & bucketing | No Python loops — scales to millions of rows |
| Multicollinearity | Correlation matrix + target comparison | Keeps the feature more predictive of `Customer_LTV` |
| Output contract | Pandera schema (`lazy=True`) | Prevents silent data corruption downstream |

This dataset is now ready to be handed to a machine learning estimator — clean, contractually
validated, and free of synthetic bias or collinearity traps.
